# Рабочие профессии: пять поисковых экспериментов

Русскоязычный профиль содержит роли, оборудование, графики, условия и исправления ввода.
C1 — текст, C2 — ограниченные сигналы, C3 — приоритет графика, C4 — понятия, C5 — исправления.
Измените профиль полей и коэффициенты, повторите ячейку сборки. Просмотр DSL не делает HTTP.
OpenSearch здесь не устанавливается. Fixture-тесты проверяют передачу запросов, но не исполняют Painless.
Авторизация берётся из окружения kernel (`OPENSEARCH_AUTHORIZATION`). Не вставляйте её в notebook.


In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/main.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

from src.components.reporting.exporters.csv import SearchCSVExporter
from src.components.reporting.exporters.json import JSONExporter
from src.experiments.batch.search import BatchSearchExperiment

## Единственный блок параметров

Имена полей ниже — пример, не обнаруженный mapping. Укажите только реально существующие
single-valued поля с doc values. Без метаданных выберите `variants = ["c1", "c4", "c5"]`
и удалите override `ranking_fields`. Для C2 можно выбрать `freshness-only` без quality.
После изменения **повторите compose**, а затем instantiate: Hydra может материализовать
внешние интерполяции в cfg; редактирование ранее разрешённого cfg не гарантирует новую связь.


In [ ]:
source_overrides = [
    "connection.url=http://localhost:9200",
    "jobboard.source.index=vacancies",
    "jobboard.source.fields=[title,description]",
    "jobboard.source.title_field=title",
    "+jobboard.source.ranking_fields={published_at:{field:published_at,type:date},quality:{field:card_completeness,type:numeric},category:{field:schedule_category,type:keyword},stable_id:{field:vacancy_id,type:keyword}}",
]
variants = ["c1", "c2", "c3", "c4", "c5"]
now_ms = int(time.time() * 1000)  # фиксируется на всю сравнительную сессию
text = "комплектовшик ТСД без опыта"
experiment_overrides = []
# Например: ["jobboard/lexical/research/ablation@_global_=no-aliases"]
# Или: ["jobboard/lexical/ranking@jobboard.rankings.signals=freshness-only"]
input_path = ROOT / "examples/blue-collar/queries.csv"
output_dir = ROOT / "outputs" / f"work-research-{now_ms}"

In [ ]:
with initialize_config_dir(config_dir=str(ROOT / "configs"), version_base="1.3"):
    cfg = compose(
        config_name="main",
        overrides=[
            "~experiments/vacancy-search/interactive@experiment",
            "+experiments/vacancy-search/jobboard/research@experiment=001-compare",
            *source_overrides,
            f"jobboard.session.now_ms={now_ms}",
            "experiment.variants=[" + ",".join(variants) + "]",
            "experiment.exporters=[]",
            *experiment_overrides,
        ],
    )
    experiment = instantiate(cfg.experiment, _convert_="all")
print("Собраны варианты:", experiment.variants)

## До поиска: что понято и что будет отправлено

Hints содержат span и rule_id; rewrite_candidates показывают принятые и отвергнутые
замены. Причина unsupported_salary_constraint означает, что диапазон зарплаты не
реализован как фильтр. Исходный запрос сохраняется. Тут нет сетевого вызова.


In [ ]:
plans = experiment.preview(text)
for variant, plan in plans.items():
    print(variant)
    print(
        json.dumps(
            {"understanding": plan.understanding, "request_body": plan.body},
            ensure_ascii=False,
            indent=2,
        )
    )

## Один запрос — все выбранные варианты

Эта ячейка делает HTTP. В таблице score не всегда имеет один смысл: C2 — новый script
score, C3 — lexical score плюс отдельный sort key, rescore — суммарная оценка.
Значения F/Q можно проверить по полям и params; компонента phrase score не выдумывается.


In [ ]:
artifact = experiment.compare(text)
artifact

In [ ]:
artifact.rank_changes()

## Готовые подборки для обсуждения

Сначала сравните роли и условия, затем ошибки. `query_sets` — быстрый старт;
полный CSV содержит более 200 проверенных локально входов. Корректный parse не
доказывает качество удалённого индекса. Неизвестные поля/запрет script требуют исправить
конфиг, а не считать пустую выдачу успехом.


In [ ]:
query_sets = {
    "склад": ["грузчик", "водитель погрузчика", "комплектовщик ТСД", "кладовщик WMS"],
    "водители": [
        "водитель категории B",
        "водитель категории CE",
        "пеший курьер",
        "курьер на своём автомобиле",
    ],
    "производство": ["оператор ЧПУ", "наладчик ЧПУ", "сварщик НАКС", "токарь 5 разряд"],
    "торговля": [
        "кассир",
        "продавец кассир",
        "работник торгового зала",
        "мерчандайзер",
    ],
    "условия": [
        "грузчик ночные смены",
        "бетонщик вахта 30/30",
        "охранник без лицензии",
        "водитель от 100000 на руки",
    ],
    "ошибки ввода": [
        "вадитель категории C",
        "rkfljdobr Москва",
        "svarshik вахта",
        "e,jhobr офиса",
    ],
}
query_sets

## Пакет и атомарное сохранение

Запуск всей матрицы делает `число запросов × число вариантов` поисков. Результаты
хранятся в памяти. Экспорт только после полного успеха, прежний файл не перезаписывается.
Реальные тексты/документы будут в артефакте; credentials и connection config — нет.


In [ ]:
from src.experiments.batch.search import read_queries

queries = read_queries(input_path)
batch_artifact = experiment.compare_queries(queries)
output_dir.mkdir(parents=True, exist_ok=True)
JSONExporter(str(output_dir / "research.json")).export(batch_artifact)
print("Сохранено:", output_dir / "research.json")

## Прежний CSV для одного варианта

Это прежний формат spec 005 без sort/parse diagnostics. Для полной диагностики
используйте research.json выше. Выбор варианта использует тот же SearchExperiment.


In [ ]:
single = BatchSearchExperiment(experiment.searches[variants[0]], str(input_path))
SearchCSVExporter(str(output_dir / "searches.csv")).export(single.run())